# 8.1 量化 (Quantization)

量化通过降低模型参数的数值精度来减少模型大小和加速推理，是最实用的模型压缩技术。

本节涵盖：
- 量化基础（对称/非对称、逐层/逐通道/逐组）
- 训练后量化（PTQ）
- 量化感知训练（QAT）
- GPTQ：基于二阶信息的4-bit量化
- AWQ：激活感知权重量化
- SmoothQuant：激活-权重难度迁移
- FP8与低精度训练
- 量化格式与部署（GGUF/GPTQ格式）

## 1. 量化基础

### 量化公式
- **量化**：$q = \text{round}(x / s + z)$
- **反量化**：$\hat{x} = (q - z) \cdot s$
- **Scale**：$s = (x_{max} - x_{min}) / (q_{max} - q_{min})$

### 量化粒度
| 粒度 | 精度 | 内存开销 | 适用场景 |
|------|------|---------|----------|
| 逐张量(per-tensor) | 低 | 最小 | 简单模型 |
| 逐通道(per-channel) | 高 | 中 | 卷积/线性层权重 |
| 逐组(per-group) | 最高 | 较大 | LLM 4-bit量化 |

### 对称 vs 非对称
- **对称**：zero_point=0，适用于权重（近似对称分布）
- **非对称**：zero_point≠0，适用于激活值（ReLU后非负）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

def symmetric_quantize(tensor, n_bits=8):
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -(2 ** (n_bits - 1))
    max_val = tensor.abs().max()
    scale = max_val / q_max
    quantized = torch.clamp(torch.round(tensor / scale), q_min, q_max).to(torch.int8)
    dequantized = quantized.float() * scale
    return quantized, dequantized, scale

def asymmetric_quantize(tensor, n_bits=8):
    q_max = 2 ** n_bits - 1
    x_min, x_max = tensor.min(), tensor.max()
    scale = (x_max - x_min) / q_max
    zero_point = torch.round(-x_min / scale)
    quantized = torch.clamp(torch.round(tensor / scale + zero_point), 0, q_max).to(torch.uint8)
    dequantized = (quantized.float() - zero_point) * scale
    return quantized, dequantized, scale, zero_point

def per_channel_quantize(tensor, n_bits=8):
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -(2 ** (n_bits - 1))
    max_val = tensor.abs().max(dim=1, keepdim=True)[0]
    scale = max_val / q_max
    quantized = torch.clamp(torch.round(tensor / scale), q_min, q_max).to(torch.int8)
    dequantized = quantized.float() * scale
    return quantized, dequantized, scale

def per_group_quantize(tensor, group_size=128, n_bits=4):
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -(2 ** (n_bits - 1))
    orig_shape = tensor.shape
    tensor_2d = tensor.reshape(-1, group_size)
    max_val = tensor_2d.abs().max(dim=1, keepdim=True)[0]
    scale = max_val / q_max
    quantized = torch.clamp(torch.round(tensor_2d / scale), q_min, q_max).to(torch.int8)
    dequantized = (quantized.float() * scale).reshape(orig_shape)
    return quantized, dequantized, scale

W = torch.randn(256, 512)

print('=== Quantization Granularity Comparison ===')
print(f'Original: FP32, {W.numel() * 4 / 1e6:.2f} MB')

q_sym, dq_sym, s_sym = symmetric_quantize(W, n_bits=8)
q_asym, dq_asym, s_asym, zp_asym = asymmetric_quantize(W, n_bits=8)
q_ch, dq_ch, s_ch = per_channel_quantize(W, n_bits=8)
q_grp, dq_grp, s_grp = per_group_quantize(W, group_size=128, n_bits=4)

results = [
    ('Symmetric INT8', (W - dq_sym).abs().mean() / W.abs().mean(), W.numel() / 1e6),
    ('Asymmetric INT8', (W - dq_asym).abs().mean() / W.abs().mean(), W.numel() / 1e6),
    ('Per-channel INT8', (W - dq_ch).abs().mean() / W.abs().mean(), W.numel() / 1e6),
    ('Per-group INT4', (W - dq_grp).abs().mean() / W.abs().mean(), W.numel() / 2 / 1e6),
]

print(f'{"Method":<20} {"Relative Error":>15} {"Size (MB)":>10}')
print('-' * 48)
for name, err, size in results:
    print(f'{name:<20} {err:>15.4f} {size:>10.2f}')

print(f'\nKey: Per-group quantization (used in GPTQ/AWQ) achieves the best 4-bit accuracy.')
print(f'Per-channel is standard for INT8; per-group is needed for sub-byte quantization.')

## 2. 量化感知训练（QAT）

**核心思想**：训练时模拟量化效果，让模型适应精度损失。

### 伪量化（Fake Quantization）
- 前向传播：$\hat{x} = \text{round}(x/s) \cdot s$（量化+反量化）
- 反向传播：$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial \hat{x}}$（STE直通估计器）

### STE的直觉
取整操作不可导，STE假设量化前后的梯度相同，即"直通"。虽然不精确，但实践中效果良好。

### 产业实践
- PyTorch: `torch.quantization`
- TensorFlow: `tf.quantization`
- QAT通常比PTQ精度高1-3%，但需要训练成本

In [ ]:
class FakeQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, scale, q_min, q_max):
        x_q = torch.clamp(torch.round(x / scale), q_min, q_max)
        ctx.save_for_backward(x, scale)
        ctx.q_min = q_min
        ctx.q_max = q_max
        return x_q * scale

    @staticmethod
    def backward(ctx, grad_output):
        x, scale = ctx.saved_tensors
        x_q = torch.clamp(torch.round(x / scale), ctx.q_min, ctx.q_max) * scale
        in_range = (x >= ctx.q_min * scale) & (x <= ctx.q_max * scale)
        grad_input = grad_output * in_range.float()
        return grad_input, None, None, None

class QATLinear(nn.Module):
    def __init__(self, in_features, out_features, n_bits=8):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.n_bits = n_bits
        self.q_max = 2 ** (n_bits - 1) - 1
        self.q_min = -(2 ** (n_bits - 1))
        self.weight_scale = None
        self.act_scale = None
        self.quantize_weights = True
        self.quantize_activations = True

    def forward(self, x):
        if self.weight_scale is None:
            self.weight_scale = self.linear.weight.abs().max() / self.q_max
        if self.act_scale is None:
            self.act_scale = x.abs().max() / self.q_max
        w = self.linear.weight
        if self.quantize_weights:
            w = FakeQuantize.apply(w, self.weight_scale, self.q_min, self.q_max)
        if self.quantize_activations:
            x = FakeQuantize.apply(x, self.act_scale, 0, 2**self.n_bits - 1)
        return F.linear(x, w, self.linear.bias)

class QATModel(nn.Module):
    def __init__(self, d=128, n_classes=10, n_bits=8):
        super().__init__()
        self.fc1 = QATLinear(d, 64, n_bits)
        self.fc2 = QATLinear(64, n_classes, n_bits)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

qat_model = QATModel(d=128, n_classes=10, n_bits=8)
optimizer = torch.optim.Adam(qat_model.parameters(), lr=1e-3)

x = torch.randn(16, 128)
y = torch.randint(0, 10, (16,))

print('=== QAT Training ===')
for epoch in range(10):
    out = qat_model(x)
    loss = F.cross_entropy(out, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    qat_model.fc1.weight_scale = None
    qat_model.fc1.act_scale = None
    qat_model.fc2.weight_scale = None
    qat_model.fc2.act_scale = None
    if (epoch + 1) % 3 == 0:
        acc = (out.argmax(1) == y).float().mean()
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, acc={acc:.2%}')

print(f'\nKey: QAT inserts fake-quantize nodes in forward pass.')
print(f'STE allows gradients to flow through non-differentiable round() operation.')
print(f'Result: model learns to be robust to quantization noise.')

## 3. GPTQ：基于二阶信息的4-bit量化

**GPTQ** (Frantar et al., 2022) 是目前最流行的LLM 4-bit量化方法。

### 核心思想
逐层量化权重，使用Hessian矩阵信息最小化量化误差：
$$\arg\min_{W_q} \|WX - W_qX\|^2$$

### 算法流程
1. 对每层，获取校准数据X的Hessian：$H = 2X X^T$
2. 按行独立量化权重
3. 逐列量化，每量化一列后，用Hessian信息调整未量化列
4. 公式：$\delta_F = -\frac{w_q - w}{[H_F^{-1}]_{pp}} \cdot H_F^{-1}_{:,p}$

### 关键特性
- **逐列量化+补偿**：量化一列后立即补偿误差到其他列
- **Lazy Batch**：批量处理列以提高效率
- **Cholesky分解**：稳定Hessian逆的计算

**产业应用**：AutoGPTQ、vLLM、HuggingFace Transformers集成

In [ ]:
def gptq_quantize(W, X, n_bits=4, block_size=128):
    d_row, d_col = W.shape
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -(2 ** (n_bits - 1))
    H = 2 * X @ X.T + 1e-6 * torch.eye(d_col, device=W.device)
    H_inv = torch.linalg.inv(H)
    Losses = torch.zeros_like(W)
    Q = torch.zeros_like(W)
    W_copy = W.clone()

    for i in range(0, d_col, block_size):
        block_end = min(i + block_size, d_col)
        for j in range(i, block_end):
            w = W_copy[:, j]
            d = H_inv[j, j]
            scale = w.abs().max() / q_max
            q = torch.clamp(torch.round(w / scale), q_min, q_max)
            Q[:, j] = q
            err1 = (w - q * scale) / d
            W_copy[:, j:] += err1.unsqueeze(1) * H_inv[j, j:].unsqueeze(0)
            Losses[:, j] = (w - q * scale) ** 2 / d

    scale_per_col = W.abs().max(dim=0)[0] / q_max
    W_dequant = Q * scale_per_col.unsqueeze(0)
    return Q, W_dequant, scale_per_col

W = torch.randn(128, 256)
X = torch.randn(256, 32)

print('=== GPTQ Quantization ===')
Q_gptq, W_dq_gptq, scales = gptq_quantize(W, X, n_bits=4, block_size=64)
gptq_error = (W - W_dq_gptq).norm() / W.norm()

q_max_naive = 2 ** 3 - 1
scale_naive = W.abs().max(dim=1, keepdim=True)[0] / q_max_naive
Q_naive = torch.clamp(torch.round(W / scale_naive), -(2**3), q_max_naive)
W_dq_naive = Q_naive * scale_naive
naive_error = (W - W_dq_naive).norm() / W.norm()

print(f'Naive per-row INT4: relative error = {naive_error:.4f}')
print(f'GPTQ INT4:          relative error = {gptq_error:.4f}')
print(f'Improvement: {(naive_error - gptq_error) / naive_error:.1%}')

orig_out = X.T @ W.T
gptq_out = X.T @ W_dq_gptq.T
naive_out = X.T @ W_dq_naive.T
print(f'\nOutput error (vs FP32):')
print(f'  Naive: {(orig_out - naive_out).norm() / orig_out.norm():.4f}')
print(f'  GPTQ:  {(orig_out - gptq_out).norm() / orig_out.norm():.4f}')

print(f'\nKey: GPTQ uses Hessian information to compensate quantization error.')
print(f'Each column quantization adjusts remaining columns to minimize total error.')

## 4. AWQ：激活感知权重量化

**AWQ** (Lin et al., 2023) 发现：权重通道的重要性不均匀，激活值大的通道更关键。

### 核心观察
- 仅约1%的权重通道对模型精度至关重要
- 这些关键通道对应于激活值较大的通道
- 保护这些通道可以大幅减少量化损失

### 方法
1. **重要性评分**：$\text{importance}_j = \max(|X_j|) \cdot \max(|W_{:,j}|)$
2. **通道缩放**：对重要通道乘以缩放因子$s > 1$，等效于提高其精度
3. **最优缩放**：通过搜索找到使量化误差最小的缩放因子

### AWQ vs GPTQ
| 维度 | GPTQ | AWQ |
|------|------|-----|
| 原理 | Hessian补偿 | 重要性感知缩放 |
| 校准数据 | 需要 | 需要 |
| 速度 | 量化慢，推理快 | 量化快，推理快 |
| 精度 | 高 | 相当 |
| 泛化性 | 对分布外数据略差 | 更好 |

In [ ]:
def awq_quantize(W, X, n_bits=4, alpha=0.5, search_steps=20):
    d_out, d_in = W.shape
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -(2 ** (n_bits - 1))
    act_scales = X.abs().max(dim=1)[0]
    weight_scales = W.abs().max(dim=0)[0]
    importance = act_scales * weight_scales
    best_scale = torch.ones(d_in)
    best_error = float('inf')

    for step in range(search_steps):
        s = 1.0 + (step / search_steps) * alpha
        top_k = int(0.01 * d_in)
        important_idx = importance.topk(top_k).indices
        scale = torch.ones(d_in)
        scale[important_idx] = s
        W_scaled = W * scale.unsqueeze(0)
        per_ch_max = W_scaled.abs().max(dim=0)[0]
        ch_scale = per_ch_max / q_max
        Q = torch.clamp(torch.round(W_scaled / ch_scale.unsqueeze(0)), q_min, q_max)
        W_dq = Q * ch_scale.unsqueeze(0) / scale.unsqueeze(0)
        error = (W - W_dq).norm().item()
        if error < best_error:
            best_error = error
            best_scale = scale.clone()

    W_scaled = W * best_scale.unsqueeze(0)
    per_ch_max = W_scaled.abs().max(dim=0)[0]
    ch_scale = per_ch_max / q_max
    Q = torch.clamp(torch.round(W_scaled / ch_scale.unsqueeze(0)), q_min, q_max)
    W_dq = Q * ch_scale.unsqueeze(0) / best_scale.unsqueeze(0)
    return Q, W_dq, best_scale, importance

W = torch.randn(128, 256)
X = torch.randn(32, 256)

Q_awq, W_dq_awq, best_scale, importance = awq_quantize(W, X, n_bits=4)
awq_error = (W - W_dq_awq).norm() / W.norm()

q_max_naive = 2 ** 3 - 1
scale_naive = W.abs().max(dim=0, keepdim=True)[0] / q_max_naive
Q_naive = torch.clamp(torch.round(W / scale_naive), -(2**3), q_max_naive)
W_dq_naive = Q_naive * scale_naive
naive_error = (W - W_dq_naive).norm() / W.norm()

print('=== AWQ (Activation-Aware Weight Quantization) ===')
print(f'Naive INT4: relative error = {naive_error:.4f}')
print(f'AWQ INT4:   relative error = {awq_error:.4f}')
print(f'Improvement: {(naive_error - awq_error) / naive_error:.1%}')
top_5 = importance.topk(5)
print(f'\nTop-5 important channels: {top_5.indices.tolist()}')
print(f'Their importance scores: {[f"{s:.2f}" for s in top_5.values.tolist()]}')
print(f'Best scale range: [{best_scale.min():.2f}, {best_scale.max():.2f}]')

print(f'\nKey: AWQ protects important channels (high activation × weight) with scaling.')
print(f'This is more robust than GPTQ for out-of-distribution inputs.')

## 5. SmoothQuant：激活-权重难度迁移

**SmoothQuant** (Xiao et al., 2023) 解决W8A8量化中激活值难以量化的问题。

### 核心问题
LLM中激活值存在离群值(outliers)，导致INT8量化精度损失严重。

### 解决方案
通过数学等价变换，将量化难度从激活值迁移到权重：
$$Y = XW = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W)$$

缩放因子：$s_j = \max(|X_{:,j}|)^\alpha / \max(|W_{j,:}|)^{1-\alpha}$

- $\alpha$越大，越多难度迁移到权重
- 通常$\alpha=0.5$是最佳平衡点

### 产业应用
- NVIDIA TensorRT-LLM集成SmoothQuant
- 支持W8A8的零精度损失量化

In [ ]:
def smoothquant_transform(W, X, alpha=0.5):
    act_scales = X.abs().max(dim=0)[0]
    weight_scales = W.abs().max(dim=0)[0]
    smooth_scale = torch.pow(act_scales / (weight_scales + 1e-8), alpha)
    smooth_scale = torch.clamp(smooth_scale, min=1e-5)
    W_smooth = W * smooth_scale.unsqueeze(0)
    x_smooth = X / smooth_scale.unsqueeze(0)
    return W_smooth, x_smooth, smooth_scale

W = torch.randn(256, 512)
X = torch.randn(32, 256)
X[:, :10] += 50

print('=== SmoothQuant ===')
print(f'Activation outlier channels: first 10 channels have 50x magnitude')

act_scales_before = X.abs().max(dim=0)[0]
print(f'Before smooth: act_max range = [{act_scales_before.min():.1f}, {act_scales_before.max():.1f}]')

for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    W_s, X_s, scale = smoothquant_transform(W, X, alpha=alpha)
    act_max_after = X_s.abs().max(dim=0)[0]
    w_max_after = W_s.abs().max(dim=0)[0]
    act_ratio = act_max_after.max() / act_max_after.min()
    orig_out = X @ W.T
    smooth_out = X_s @ W_s.T
    error = (orig_out - smooth_out).norm() / orig_out.norm()
    print(f'  alpha={alpha:.1f}: act_ratio={act_ratio:.1f}, w_max={w_max_after.max():.2f}, error={error:.6f}')

W_s, X_s, scale = smoothquant_transform(W, X, alpha=0.5)
act_scales_after = X_s.abs().max(dim=0)[0]
print(f'\nAfter smooth (alpha=0.5): act_max range = [{act_scales_after.min():.1f}, {act_scales_after.max():.1f}]')
print(f'Smooth scale range: [{scale.min():.3f}, {scale.max():.3f}]')

print(f'\nKey: SmoothQuant reduces activation outlier magnitude by transferring difficulty to weights.')
print(f'alpha=0.5 is typically optimal — equal balance between activation and weight quantization.')

## 6. FP8与低精度训练

**FP8** (E4M3/E5M2) 是NVIDIA H100/Blackwell支持的原生8-bit浮点格式。

### FP8格式
| 格式 | 符号 | 指数 | 尾数 | 范围 | 精度 | 用途 |
|------|------|------|------|------|------|------|
| E4M3 | 1 | 4 | 3 | ±448 | 高 | 前向传播 |
| E5M2 | 1 | 5 | 2 | ±57344 | 低 | 反向传播/梯度 |

### FP8训练流程
1. 前向：权重和激活用E4M3
2. 反向：梯度用E5M2
3. 动态缩放：每层维护scale factor
4. 延迟缩放：用前一步的max值缩放当前步

### 与INT8的区别
- FP8保留浮点动态范围，不需要zero_point
- FP8更适合训练（梯度有正负、范围大）
- INT8更适合推理（权重分布较均匀）

**产业应用**：NVIDIA Transformer Engine、Meta FP8训练

In [ ]:
class FP8Emulator:
    def __init__(self, format='E4M3'):
        self.format = format
        if format == 'E4M3':
            self.exp_bits = 4
            self.man_bits = 3
            self.max_exp = 8
            self.bias = 7
        else:
            self.exp_bits = 5
            self.man_bits = 2
            self.max_exp = 16
            self.bias = 15

    def quantize(self, tensor):
        max_val = tensor.abs().max()
        scale = max_val / self._max_representable()
        scaled = tensor / scale
        quantized = self._fake_fp8_round(scaled)
        return quantized * scale, scale

    def _max_representable(self):
        if self.format == 'E4M3':
            return 448.0
        else:
            return 57344.0

    def _fake_fp8_round(self, x):
        max_val = self._max_representable()
        x = torch.clamp(x, -max_val, max_val)
        mantissa_scale = 2 ** self.man_bits
        x_scaled = x * mantissa_scale
        x_rounded = torch.round(x_scaled) / mantissa_scale
        return x_rounded

fp8_e4m3 = FP8Emulator('E4M3')
fp8_e5m2 = FP8Emulator('E5M2')

W = torch.randn(128, 256)
grad = torch.randn(128, 256) * 0.01

print('=== FP8 Emulation ===')
W_e4m3, scale_e4m3 = fp8_e4m3.quantize(W)
W_e5m2, scale_e5m2 = fp8_e5m2.quantize(W)
grad_e5m2, grad_scale = fp8_e5m2.quantize(grad)

e4m3_error = (W - W_e4m3).norm() / W.norm()
e5m2_error = (W - W_e5m2).norm() / W.norm()
grad_error = (grad - grad_e5m2).norm() / grad.norm()

print(f'E4M3 (forward): relative error = {e4m3_error:.4f}, scale = {scale_e4m3:.4f}')
print(f'E5M2 (forward): relative error = {e5m2_error:.4f}, scale = {scale_e5m2:.4f}')
print(f'E5M2 (gradient): relative error = {grad_error:.4f}, scale = {grad_scale:.6f}')
print(f'\nE4M3 has higher precision (3 mantissa bits) but smaller range (±448)')
print(f'E5M2 has lower precision (2 mantissa bits) but larger range (±57344)')
print(f'\nKey: FP8 enables training with 8-bit floating point — 2x memory savings vs FP16.')
print(f'E4M3 for forward (needs precision), E5M2 for backward (needs range for gradients).')

## 7. 量化格式与部署

### 主流量化格式
| 格式 | 量化方法 | 文件格式 | 推理引擎 | 适用场景 |
|------|---------|---------|---------|----------|
| GPTQ | 4-bit权重量化 | .pt/.safetensors | vLLM/AutoGPTQ | GPU推理 |
| GGUF | 2-8-bit量化 | .gguf | llama.cpp | CPU/混合推理 |
| AWQ | 4-bit权重量化 | .pt/.safetensors | vLLM/AWQ | GPU推理 |
| FP8 | 8-bit浮点 | .safetensors | TensorRT-LLM | H100训练/推理 |
| ONNX Quant | INT8 | .onnx | ONNX Runtime | 通用部署 |

### GGUF量化类型
- **Q4_0**: 4-bit量化，block size 32
- **Q4_K_M**: 4-bit量化，部分5-bit，推荐
- **Q5_K_M**: 5-bit量化，推荐
- **Q8_0**: 8-bit量化，几乎无损

### 部署选择指南
- **GPU + 高吞吐**: GPTQ/AWQ + vLLM
- **CPU/边缘设备**: GGUF + llama.cpp
- **H100集群**: FP8 + TensorRT-LLM
- **通用部署**: ONNX INT8

In [ ]:
class QuantizationBenchmark:
    def __init__(self, d_model=256, d_ff=512, seq_len=128):
        self.d_model = d_model
        self.d_ff = d_ff
        self.seq_len = seq_len
        self.W_q = torch.randn(d_model, d_ff)
        self.W_k = torch.randn(d_model, d_ff)
        self.W_v = torch.randn(d_model, d_ff)
        self.W_o = torch.randn(d_ff, d_model)
        self.X = torch.randn(seq_len, d_model)

    def benchmark_format(self, name, quantize_fn, n_bits):
        W_q_dq = quantize_fn(self.W_q)
        W_k_dq = quantize_fn(self.W_k)
        W_v_dq = quantize_fn(self.W_v)
        W_o_dq = quantize_fn(self.W_o)
        with torch.no_grad():
            Q = self.X @ W_q_dq
            K = self.X @ W_k_dq
            V = self.X @ W_v_dq
            attn = F.softmax(Q @ K.T / (self.d_ff ** 0.5), dim=-1)
            out = attn @ V @ W_o_dq
        fp32_out = self.X @ self.W_q @ self.W_k.T @ self.X.T
        error = (self.X @ W_q_dq - self.X @ self.W_q).norm() / (self.X @ self.W_q).norm()
        size_mb = sum(w.numel() for w in [self.W_q, self.W_k, self.W_v, self.W_o]) * n_bits / 8 / 1e6
        return {'name': name, 'bits': n_bits, 'size_mb': size_mb, 'error': error.item()}

bench = QuantizationBenchmark(d_model=256, d_ff=512, seq_len=64)

def fp32_fn(w):
    return w
def int8_fn(w):
    q_max = 127
    scale = w.abs().max() / q_max
    return torch.clamp(torch.round(w / scale), -q_max, q_max) * scale
def int4_fn(w):
    q_max = 7
    scale = w.abs().max() / q_max
    return torch.clamp(torch.round(w / scale), -q_max, q_max) * scale
def fp8_fn(w):
    fp8 = FP8Emulator('E4M3')
    result, _ = fp8.quantize(w)
    return result

results = [
    bench.benchmark_format('FP32', fp32_fn, 32),
    bench.benchmark_format('FP16', lambda w: w.half().float(), 16),
    bench.benchmark_format('FP8 E4M3', fp8_fn, 8),
    bench.benchmark_format('INT8', int8_fn, 8),
    bench.benchmark_format('INT4', int4_fn, 4),
]

print('=== Quantization Format Benchmark ===')
print(f'{"Format":<12} {"Bits":>5} {"Size (MB)":>10} {"Error":>10}')
print('-' * 40)
for r in results:
    print(f'{r["name"]:<12} {r["bits"]:>5} {r["size_mb"]:>10.2f} {r["error"]:>10.4f}')

print(f'\nKey: FP8 provides 4x compression with minimal error (better than INT8).')
print(f'INT4 provides 8x compression but needs GPTQ/AWQ to maintain accuracy.')
print(f'Choose format based on hardware, accuracy requirements, and deployment target.')

## 8. 量化总结

| 方法 | 精度 | 速度 | 适用场景 | 产业工具 |
|------|------|------|----------|----------|
| PTQ INT8 | 中 | 最快 | 快速部署 | PyTorch Quantization |
| QAT | 最高 | 慢 | 极致精度 | PyTorch QAT |
| GPTQ INT4 | 高 | 快 | GPU 4-bit | AutoGPTQ, vLLM |
| AWQ INT4 | 高 | 快 | GPU 4-bit | vLLM, AWQ |
| SmoothQuant | 高 | 快 | W8A8 | TensorRT-LLM |
| FP8 | 高 | 快 | H100训练/推理 | Transformer Engine |
| GGUF | 中-高 | 中 | CPU推理 | llama.cpp |

**选择建议**：
- 追求极致压缩（4-bit）→ GPTQ或AWQ
- 追求W8A8零损失 → SmoothQuant
- 追求训练加速 → FP8
- CPU/边缘部署 → GGUF
- 快速原型 → PTQ INT8